# 🕶️ Godfather JEPA: Autonomous Visual Causal World Model with Dual-Store Memory

This turnkey Colab notebook trains and evaluates the **Godfather JEPA** agent on Crafter.

### Key Innovations:
1. **Tabula-Rasa Visual Causal JEPA**: Learns purely from raw RGB pixels ( 	imes 64 	imes 3$) and scalar rewards ( \in \mathbb{R}$) with Decoupled VICReg (Var(z) >= 1.0).
2. **Two-Tier Complementary Learning System (CLS)**:
   - *Fast Online Episodic Store (Hippocampus)*: Sub-millisecond single-trial transition recording, item deltas, and visual odometry tracking.
   - *Slow Offline Consolidated Bank (Neocortex)*: Four Iron Rules (=1$ rare milestone preservation, precondition attribution guard, spatial binding, vitals isolation).
3. **Recursive Upstream Budgeting**: Generic recursive DFS budgeting resolves the 87.5% wood deficit trap ((	ext{Pickaxe}) = 2	ext{ (table)} + 1	ext{ (pick)} = 3	ext{ Wood}$).
4. **Visual-Inertial Topological Anchor**: Closed-loop path reversal vectors return the agent to off-screen crafting tables.
5. **Homeostatic Priority Arbiter**: Preemptively overrides subgoals to drink when thirst $\le 3.0$ and readies defenses before dusk.

### Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/learn-a-world")
(DRIVE_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "godfather_runs").mkdir(parents=True, exist_ok=True)

print("Google Drive workspace mounted successfully at:", DRIVE_DIR)


### Cell 2: Fetch / Clone Latest Repository from GitHub

In [ ]:
%cd /content
import os
if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git
%cd /content/learn-a-world
!git fetch origin
!git checkout main
!git pull origin main
!git log -1 --oneline


### Cell 3: Install Dependencies & Check GPU Acceleration

In [ ]:
%cd /content/learn-a-world
!pip install -q crafter gymnasium pytest

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi


### Cell 4: Strict Architectural Isolation Audit & Unit Test Suite

In [ ]:
%cd /content/learn-a-world
# 1. Verify 100% legacy isolation (0 legacy imports)
!python3 godfather_jepa/verification/ast_audit.py

# 2. Run unit test suite
!pytest godfather_jepa/tests/test_causal_and_budgeting.py godfather_jepa/tests/test_odometry.py -v


### Cell 5: Setup Checkpoints & Drive Sequence Shards

In [ ]:
import os
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/learn-a-world")
CKPT_PATH = DRIVE_DIR / "checkpoints/jepa_crafter_v3_state_readout_v2.pt"
SHARD_DIR = DRIVE_DIR / "data/sequences/"

print(f"Vision Checkpoint exists: {CKPT_PATH.exists()}")
shard_files = list(SHARD_DIR.glob("*.npz")) if SHARD_DIR.exists() else []
print(f"Found {len(shard_files)} sequence shards in {SHARD_DIR}")


### Cell 6: Offline Memory Consolidation (Semantic Bank Pre-Bake)

In [ ]:
%cd /content/learn-a-world
!python3 godfather_jepa/memory/consolidate_offline.py \n  --shard-dir /content/drive/MyDrive/learn-a-world/data/sequences/ \n  --out-bank /content/drive/MyDrive/learn-a-world/checkpoints/godfather_memory_bank.pt \n  --max-episodes 500


### Cell 7: Turnkey Training & Online Self-Play (GPU)

In [ ]:
%cd /content/learn-a-world
!python3 godfather_jepa/training/train.py \n  --device cuda \n  --episodes 100 \n  --checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_state_readout_v2.pt \n  --shard-dir /content/drive/MyDrive/learn-a-world/data/sequences/ \n  --out-dir /content/drive/MyDrive/learn-a-world/godfather_runs/


### Cell 8: Locked Benchmark Evaluation (Held-out seeds 2050..2075)

In [ ]:
%cd /content/learn-a-world
import torch
from godfather_jepa.training.train import GodfatherJEPAPipeline

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline = GodfatherJEPAPipeline(device=device)
pipeline.load_checkpoint("/content/drive/MyDrive/learn-a-world/godfather_runs/godfather_jepa_final.pt")

results = pipeline.evaluate(seeds=list(range(2050, 2075)))
print("=" * 50)
print(f"GODFATHER JEPA OFFICIAL BENCHMARK REPORT:")
print(f"Crafter Score:        {results['crafter_score']:.2f}%")
print(f"Mean Episode Reward:  {results['mean_reward']:.2f}")
print(f"Mean Survival Length: {results['mean_survival_length']:.1f} steps")
print("
Achievement Unlock Rates:")
for k, v in sorted(results['achievements'].items()):
    if v > 0:
        print(f"  {k:22s}: {v * 100:.1f}%")
print("=" * 50)
